In [2]:
!pip install linearmodels -q

import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

print("✅ Ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 20.9 MB/s eta 0:00:00
Mounted at /content/drive
✅ Ready


In [ ]:
for folder in ['/MyDrive/MTA/raw_data/2020-2024/MTA_Ridership_20_24.csv', '/MyDrive/MTA/clean_data/MTA_Daily_Clean_2025.csv']:
    print(f"\n📁 {folder}")
    if os.path.exists(folder):
        for f in sorted(os.listdir(folder)):
            size = os.path.getsize(os.path.join(folder, f)) / 1e6
            print(f"   {f}  ({size:.1f} MB)")
    else:
        print("   ❌ Folder not found — check the path above")

In [3]:
df_2024 = pd.read_csv('/content/drive/MyDrive/MTA/raw_data/2020-2024/MTA_Ridership_20_24.csv')

In [4]:
df_2025 = pd.read_csv('/content/drive/MyDrive/MTA/clean_data/MTA_Daily_Clean_2025.csv')

In [5]:
print("── 2024 raw ──")
print(f"Rows: {len(df_2024):,}")
print(f"Columns: {list(df_2024.columns)}")
print()
print("── 2025 cleaned ──")
print(f"Rows: {len(df_2025):,}")
print(f"Columns: {list(df_2025.columns)}")

── 2024 raw ──
Rows: 120,855,567
Columns: ['transit_timestamp', 'transit_mode', 'station_complex_id', 'station_complex', 'borough', 'payment_method', 'fare_class_category', 'ridership', 'transfers', 'latitude', 'longitude', 'Georeference']

── 2025 cleaned ──
Rows: 35,382,811
Columns: ['station_complex_id', 'station_complex', 'borough', 'latitude', 'longitude', 'Date', 'Day_of_Week', 'transit_timestamp', 'fare_class_category', 'ridership', 'transfers', 'CRZ_Zone', 'Hour']


In [7]:
# ─────────────────────────────────────────────────────────────
# Step 3: Filter, clean, and align 2024 to match 2025 schema
# ─────────────────────────────────────────────────────────────
import re

print("Step 1/6 — Parsing timestamps...")
df_2024['transit_timestamp'] = pd.to_datetime(df_2024['transit_timestamp'])

print("Step 2/6 — Filtering to 2024 only...")
df_2024 = df_2024[df_2024['transit_timestamp'].dt.year == 2024].copy()
print(f"  Rows after filter: {len(df_2024):,}")

print("Step 3/6 — Fixing numeric types...")
df_2024['ridership']  = pd.to_numeric(df_2024['ridership'],  errors='coerce').fillna(0).astype(int)
df_2024['transfers']  = pd.to_numeric(df_2024['transfers'],  errors='coerce').fillna(0).astype(int)

print("Step 4/6 — Extracting Date, Hour, Day_of_Week features...")
df_2024['Date']        = df_2024['transit_timestamp'].dt.date
df_2024['Hour']        = df_2024['transit_timestamp'].dt.hour
df_2024['Day_of_Week'] = df_2024['transit_timestamp'].dt.day_name()

# Convert timestamp to time-only — same as what was done to 2025 in your notebook
df_2024['transit_timestamp'] = df_2024['transit_timestamp'].dt.time

print("Step 5/6 — Dropping columns not in 2025 schema...")
cols_to_drop = ['transit_mode', 'Georeference', 'payment_method']
df_2024.drop(columns=[c for c in cols_to_drop if c in df_2024.columns], inplace=True)

print("Step 6/6 — Applying CRZ_Zone geofence (same logic as 2025)...")
def assign_toll_zone(row):
    if row['borough'] != 'Manhattan':
        return 'Outer Boroughs'
    station_name = str(row['station_complex'])
    name_no_trains = re.sub(r'\(.*?\)', '', station_name)
    numbers = re.findall(r'\d+', name_no_trains)
    for num in numbers:
        if int(num) > 59:
            return 'Manhattan (Above 60th St - Outside Toll Zone)'
    return 'Manhattan (CBD Toll Zone)'

df_2024['CRZ_Zone'] = df_2024.apply(assign_toll_zone, axis=1)

# Reorder columns to exactly match 2025
col_order = ['station_complex_id', 'station_complex', 'borough', 'latitude', 'longitude',
             'Date', 'Day_of_Week', 'transit_timestamp', 'fare_class_category',
             'ridership', 'transfers', 'CRZ_Zone', 'Hour']
df_2024 = df_2024[col_order]

print("\n✅ Done! Final check:")
print(f"  Shape: {df_2024.shape}")
print(f"  Columns: {list(df_2024.columns)}")
print(f"\n2025 columns: {list(df_2025.columns)}")
print(f"Schemas match: {list(df_2024.columns) == list(df_2025.columns)}")

Step 1/6 — Parsing timestamps...
Step 2/6 — Filtering to 2024 only...
  Rows after filter: 27,012,513
Step 3/6 — Fixing numeric types...
Step 4/6 — Extracting Date, Hour, Day_of_Week features...
Step 5/6 — Dropping columns not in 2025 schema...
Step 6/6 — Applying CRZ_Zone geofence (same logic as 2025)...

✅ Done! Final check:
  Shape: (27012513, 13)
  Columns: ['station_complex_id', 'station_complex', 'borough', 'latitude', 'longitude', 'Date', 'Day_of_Week', 'transit_timestamp', 'fare_class_category', 'ridership', 'transfers', 'CRZ_Zone', 'Hour']

2025 columns: ['station_complex_id', 'station_complex', 'borough', 'latitude', 'longitude', 'Date', 'Day_of_Week', 'transit_timestamp', 'fare_class_category', 'ridership', 'transfers', 'CRZ_Zone', 'Hour']
Schemas match: True


In [8]:
# ─────────────────────────────────────────────────────────────
# Step 4: Save cleaned 2024 to Google Drive
# ─────────────────────────────────────────────────────────────

SAVE_PATH = '/content/drive/MyDrive/MTA/clean_data/mta_2024_cleaned_cj.csv'

print(f"Saving {len(df_2024):,} rows to Drive...")
print("(This will take 1–2 minutes for a ~27M row file)")

df_2024.to_csv(SAVE_PATH, index=False)

import os
size_mb = os.path.getsize(SAVE_PATH) / 1e6
print(f"\n✅ Saved! → {SAVE_PATH}")
print(f"   File size: {size_mb:.1f} MB")

Saving 27,012,513 rows to Drive...
(This will take 1–2 minutes for a ~27M row file)

✅ Saved! → /content/drive/MyDrive/MTA/clean_data/mta_2024_cleaned_cj.csv
   File size: 3541.9 MB


In [4]:
# ─────────────────────────────────────────────────────────────
# Step 5: Merge datasets + enrich with official CBD designation
# ─────────────────────────────────────────────────────────────

# ── 5a: Stack 2024 and 2025 ───────────────────────────────────
print("Merging 2024 and 2025...")

df_2024['year'] = 2024
df_2025['year'] = 2025

df = pd.concat([df_2024, df_2025], ignore_index=True)
print(f"  Combined rows: {len(df):,}")
print(f"  2024: {(df['year']==2024).sum():,}  |  2025: {(df['year']==2025).sum():,}")

# ── 5b: Load official MTA station metadata ────────────────────
print("\nLoading MTA Subway Stations file...")

# Update this path to where you uploaded the file in your Drive
STATIONS_PATH = '/content/drive/MyDrive/MTA/MTA Map/MTA_Subway_Stations_20260402.csv'

stations = pd.read_csv(STATIONS_PATH)

# Keep only what we need
stations = stations[['Complex ID', 'CBD', 'Division', 'Line', 'Daytime Routes', 'Structure']].copy()
stations.columns = ['station_complex_id', 'CBD_official', 'Division', 'Line', 'Daytime_Routes', 'Structure']

# Complex ID is an int in the stations file — match type to our panel
stations['station_complex_id'] = stations['station_complex_id'].astype(str)
df['station_complex_id']       = df['station_complex_id'].astype(str)

# One row per complex (some complexes have multiple stops — deduplicate)
stations = stations.drop_duplicates(subset='station_complex_id')

print(f"  Stations loaded: {len(stations)}")
print(f"  CBD=True stations: {stations['CBD_official'].sum()}")

# ── 5c: Join onto panel ───────────────────────────────────────
print("\nJoining station metadata...")
df = df.merge(stations, on='station_complex_id', how='left')

matched = df['CBD_official'].notna().sum()
print(f"  Rows matched: {matched:,} / {len(df):,}")

# ── 5d: Replace string-match CRZ_Zone with official CBD flag ──
# CBD_official = True  → inside congestion zone (treatment)
# CBD_official = False → outside congestion zone (control)
# NaN          → non-Manhattan / no match → outside zone

df['CBD_official'] = df['CBD_official'].fillna(False)
df['treated']      = df['CBD_official'].astype(int)  # 1 = treatment, 0 = control

# Keep old CRZ_Zone for reference but mark which source is being used
print("\n── Treatment assignment (official CBD flag) ──")
print(df.groupby('treated')['station_complex_id'].nunique()
        .rename({1: 'Treatment stations (CBD=True)', 0: 'Control stations (CBD=False)'}))

# ── 5e: Quick sanity check ────────────────────────────────────
print("\n── Sample treatment stations (CBD=True, Manhattan) ──")
sample_t = df[(df['treated']==1) & (df['borough']=='Manhattan')][['station_complex','latitude','Division']].drop_duplicates().head(8)
print(sample_t.to_string(index=False))

print("\n── Sample control stations (CBD=False, Manhattan) ──")
sample_c = df[(df['treated']==0) & (df['borough']=='Manhattan')][['station_complex','latitude','Division']].drop_duplicates().head(8)
print(sample_c.to_string(index=False))

print(f"\n✅ Master panel ready: {df.shape}")

Merging 2024 and 2025...


NameError: name 'df_2024' is not defined